# No GS

In [1]:
%%time

# Import libraries
import os
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb


from sklearn.model_selection import (
    train_test_split,
    TimeSeriesSplit,
    GridSearchCV,
    cross_val_score,
    KFold
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)


# Load dataset
df_model = pd.read_csv(
    'transformed_daily_data/daily_data_5y_30k_features1.csv'
)

# Prepare dataset

# Ensure datetime formatting
df_model['timestamp'] = pd.to_datetime(df_model['timestamp'])

# Sort dataset
df_model = df_model.sort_values(
    ['ticker', 'timestamp']
).copy()


# Next day closing price
df_model['next_close'] = (
    df_model.groupby('ticker')['close'].shift(-1)
)

# Remove final row per ticker for which next_close does not exist
df_model = df_model[
    df_model['next_close'].notna()
].copy()


# Next close change
df_model['next_close_change'] = df_model['next_close'] - df_model['close']


# Next close change PCT - This will be the target
df_model['next_close_change_pct'] = (
    (df_model['next_close'] - df_model['close']) / df_model['close']
) * 100







# Train/test split

# Train on 2021-2024 data
train_df = df_model[
    (df_model['timestamp'] >= '2021-01-01') &
    (df_model['timestamp'] < '2025-01-01')
].copy()

# Test on 2025 data
test_df = df_model[
    (df_model['timestamp'] >= '2025-01-01') &
    (df_model['timestamp'] < '2026-01-01')
].copy()


# Select features - exclude index, target, leakage columns, and features determined ot be of low quality
exclude_cols = [
    'timestamp',
    'ticker',
    'next_close',
    'next_close_change',
    'next_close_change_pct',
    'rolling_volatility',
    'volume',
    'daily_high_vs_low_pct'
]

feature_cols = [
    col for col in df_model.columns
    if col not in exclude_cols
]


X_train = train_df[feature_cols]
y_train = train_df['next_close_change_pct']

X_test = test_df[feature_cols]
y_test = test_df['next_close_change_pct']


# Define model
model = xgb.XGBRegressor(random_state=42)

# Add GridSearch (not doing it in this initial notebook, just saving my place)


# Train model
model.fit(X_train, y_train)


# Get predictinos
y_test_pred = model.predict(X_test)


# Define evaluation metrics
mae = mean_absolute_error(y_test, y_test_pred)
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
medae = median_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)




# Create results dataframe
results_df = test_df.copy()


# Add predictions
results_df['pred_next_close_change_pct'] = (y_test_pred).round(2)

# Impute error
results_df['pred_error'] = results_df['pred_next_close_change_pct'] - results_df['next_close_change_pct']

# Absolute error
results_df['abs_pred_error'] = results_df['pred_error'].abs()

# Directionally correct
results_df['directionally_correct'] = (
    np.sign(results_df['next_close_change_pct']) ==
    np.sign(results_df['pred_next_close_change_pct'])
).astype(int)

# Save results
results_df.to_csv('models_daily_results/Round_2_XGBR_Default.csv', index=False)



# Print evaluation metrics
print("\n===== REGRESSION METRICS =====")

print(f"MAE:    {mae:.4f}")
print(f"MSE:    {mse:.4f}")
print(f"RMSE:   {rmse:.4f}")
print(f"MedAE:  {medae:.4f}")
print(f"R²:     {r2:.4f}")

print(
    f"Percent directionally correct: "
    f"{results_df['directionally_correct'].mean() * 100:.2f}%"
)


===== REGRESSION METRICS =====
MAE:    1.9077
MSE:    10.6053
RMSE:   3.2566
MedAE:  1.1877
R²:     -0.0272
Percent directionally correct: 48.14%
CPU times: user 6.49 s, sys: 1.01 s, total: 7.51 s
Wall time: 5.13 s


# GridSearch for best RSME

In [ ]:
%%time

# Import libraries
import os
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from xgboost import XGBRegressor

from sklearn.model_selection import GridSearchCV



from sklearn.model_selection import (
    train_test_split,
    TimeSeriesSplit,
    GridSearchCV,
    cross_val_score,
    KFold
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)


# Load dataset
df_model = pd.read_csv(
    'transformed_daily_data/daily_data_5y_30k_features1.csv'
)

# Prepare dataset

# Ensure datetime formatting
df_model['timestamp'] = pd.to_datetime(df_model['timestamp'])

# Sort dataset
df_model = df_model.sort_values(
    ['ticker', 'timestamp']
).copy()


# Next day closing price
df_model['next_close'] = (
    df_model.groupby('ticker')['close'].shift(-1)
)

# Remove final row per ticker for which next_close does not exist
df_model = df_model[
    df_model['next_close'].notna()
].copy()


# Next close change
df_model['next_close_change'] = df_model['next_close'] - df_model['close']


# Next close change PCT - This will be the target
df_model['next_close_change_pct'] = (
    (df_model['next_close'] - df_model['close']) / df_model['close']
) * 100







# Train/test split

# Train on 2021-2024 data
train_df = df_model[
    (df_model['timestamp'] >= '2021-01-01') &
    (df_model['timestamp'] < '2025-01-01')
].copy()

# Test on 2025 data
test_df = df_model[
    (df_model['timestamp'] >= '2025-01-01') &
    (df_model['timestamp'] < '2026-01-01')
].copy()


# Select features - exclude index, target, leakage columns, and features determined ot be of low quality
exclude_cols = [
    'timestamp',
    'ticker',
    'next_close',
    'next_close_change',
    'next_close_change_pct',
    'rolling_volatility',
    'volume',
    'daily_high_vs_low_pct'
]


feature_cols = [
    col for col in df_model.columns
    if col not in exclude_cols
]


X_train = train_df[feature_cols]
y_train = train_df['next_close_change_pct']

X_test = test_df[feature_cols]
y_test = test_df['next_close_change_pct']


# Define model
model = xgb.XGBRegressor(random_state=42)

# GridSearch

# #xgbr = XGBRegressor(
#     objective="reg:squarederror",
#     random_state=42,
#     n_jobs=-1
# )


# GridSearchCV(
#     model,
#     param_grid,
#     scoring=direction_scorer,
#     cv=tscv
# )


param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [3, 5],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5],
    "reg_lambda": [1, 5]
}


# GridSearchCV(
#     model,
#     param_grid,
#     scoring='neg_root_mean_squared_error',
# )


grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=2
)

# Expanded param grid
# param_grid = {
#     "n_estimators": [100, 300, 500],
#     "max_depth": [3, 5, 7],
#     "learning_rate": [0.01, 0.05, 0.1],
#     "subsample": [0.8, 1.0],
#     "colsample_bytree": [0.8, 1.0],
#     "min_child_weight": [1, 3, 5],
#     "reg_alpha": [0, 0.01, 0.1],
#     "reg_lambda": [1, 5, 10]
# }


grid_search.fit(X_train, y_train)


best_model = grid_search.best_estimator_

print("Best params:")
print(grid_search.best_params_)

print("Best MAE:")
print(-grid_search.best_score_)

Fitting 5 folds for each of 192 candidates, totalling 960 fits
